<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c03-header.png" alt="Nextia Learning · Essential Mathematics and Statistics for AI" width="100%">

# Solution: Functions and change

**[Functions and change](https://learning.nextia-ai.com/courses/math/m04/functions-and-change/)** · Essential Mathematics and Statistics for AI · Module 4, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**Optional: check the numbers in Python.** This notebook is optional. The lesson works without code: a calculator is enough. Use the notebook to check the lesson's numbers and to redo its worked examples and tasks in Python.

**You will** write Tomás's ticket model as a Python function, measure its slope with units, and make the gap between two points on a curve smaller and smaller until the slope becomes the derivative.

| | |
|---|---|
| **Time** | About 20 minutes, after you read the lesson |
| **Needs** | Nothing to install and no account: only Python's standard library. It runs in Colab, Kaggle or any Jupyter. |
| **You should know** | How to read units such as "minutes per hundred words", from [Mathematical reading skills: Units](https://learning.nextia-ai.com/courses/math/m01/mathematical-reading-skills/#units), and how to read a decimal such as 0.01, from [Fractions, decimals and percentages](https://learning.nextia-ai.com/courses/math/m01/mathematical-reading-skills/#fractions-decimals-and-percentages). |
| **Tested** | 2026-10-07, Python 3.14.6 (standard library only), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The [lesson page](https://learning.nextia-ai.com/courses/math/m04/functions-and-change/) has the full explanations, the charts, the explorers and the knowledge checks that count toward your certificate. Read a section there, then check its numbers here.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, replace each `...` with your calculation, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C03/M04-L01-functions-and-change/functions-and-change-solution.ipynb).

## Setup: the helpers

The next cell defines three small functions that the notebook uses. You do not need to read them, but you can.

- `expect(what, yours, expected)` compares your number with the lesson's number. It says "Check passed." or if yours is too high or too low. It does not show the answer.
- `table(headers, rows)` prints a small table with aligned columns.
- `bars(labels, counts)` prints a text bar chart: one row of `█` for each label.

Run the cell. You should see `Ready.`

In [ ]:
# Helpers: check an answer without showing it, print a table, draw text bars.
import math, random, statistics

def expect(what, yours, expected, tol=0.01):
    """Compare your value with the expected one, to within `tol` for numbers."""
    if yours is ...:
        print(f"Not yet: replace ... with your calculation for {what}, then run the cell again.")
        return
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tol:
            print(f"Check passed: {what} is {yours:g}.")
        else:
            side = "too high" if yours > expected else "too low"
            print(f"Not yet: your {what}, {yours:g}, is {side}. Look at the step before it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: your {what} is {yours!r}. That is not the expected answer.")

def table(headers, rows, digits=2):
    """Print rows as a table. Floats are rounded to `digits` decimals."""
    cell = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else str(v)
    text = [[cell(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i, h in enumerate(headers)]
    print("  ".join(str(h).rjust(w) for h, w in zip(headers, widths)))
    for r in text:
        print("  ".join(v.rjust(w) for v, w in zip(r, widths)))

def bars(labels, counts, width=40):
    """Print a text bar chart: the longest bar is `width` characters."""
    top = max(counts) or 1
    size = max(len(str(label)) for label in labels)
    for label, n in zip(labels, counts):
        print(f"{str(label).rjust(size)} | {'█' * round(n / top * width)} {n}")

print("Ready.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A model is a function

A **function** is a rule that gives exactly one output for each input. Tomás's model is the rule minutes = w × length. The input is the length of a ticket, in hundreds of words. The output is the predicted handling time, in minutes. The number `w` is a **parameter**: a number inside the model that training chooses.

The next cell writes the rule as a Python function. It makes the lesson's table for `w = 6`. Run it. You should see the predictions 6, 12, 18 and 24 next to the actual minutes 7, 13, 17 and 26.

In [ ]:
tickets = ["T-3101", "T-3102", "T-3103", "T-3104"]
lengths = [1, 2, 3, 4]       # hundreds of words: the input
minutes = [7, 13, 17, 26]    # actual handling minutes

def predict(w, length):
    """Tomás's model: predicted minutes = w × length."""
    return w * length

rows = [[t, x, predict(6, x), y] for t, x, y in zip(tickets, lengths, minutes)]
table(["Ticket", "Length", "Prediction (w = 6)", "Actual minutes"], rows)

The model gives an answer for any length, also for lengths that it never saw.

> **Predict.** What does the model with `w = 6` predict for a ticket of 250 words (length 2.5)? Write it down, then run the cell.

In [ ]:
print("prediction for length 2.5:", predict(6, 2.5), "minutes")

> **Check.** You should see `15.0` minutes. This is the value that you read from the line in the lesson's first chart.

## 2. Slope: change in output per unit of input

The **slope** is the change in output divided by the change in input: Δy ÷ Δx. Its unit is output units per input unit, here minutes per hundred words.

The next cell measures the slope of the model with `w = 6` between two pairs of lengths. Run it. You should see `6.0` two times: on a straight line, the slope is the same everywhere.

In [ ]:
def slope(f, x1, x2):
    """The slope of the function f between x1 and x2: change in output ÷ change in input."""
    return (f(x2) - f(x1)) / (x2 - x1)

def model_6(length):
    return predict(6, length)

print("slope from length 2 to 4:", slope(model_6, 2, 4), "minutes per hundred words")
print("slope from length 1 to 3:", slope(model_6, 1, 3), "minutes per hundred words")

For this model, the parameter `w` is the slope. A larger `w` gives a steeper line.

> **Your turn: change one thing.** In the cell above, change `predict(6, length)` to `predict(8, length)` and run the cell again. You should see `8.0` two times. Then change it back to 6.

## 3. Curves: the slope changes

Omar wants a better ticket classifier. Agents can label more past tickets to train it, but labelling costs agent time. Priya's **learning curve** shows the accuracy of the classifier after training on x thousand labelled tickets. The values are illustrative, not measured. They follow the rule accuracy = 94 − 24 ÷ x.

Run the cell. You should see the lesson's table: 70, 82, 88 and 91% at 1, 2, 4 and 8 thousand tickets.

In [ ]:
def accuracy(x):
    """Accuracy (%) after x thousand labelled tickets: the lesson's learning curve."""
    return 94 - 24 / x

table(["Labelled tickets (thousands)", "Accuracy (%)"], [[x, accuracy(x)] for x in [1, 2, 4, 8]], digits=1)

> **Predict.** Is the slope from 1 to 2 thousand tickets larger or smaller than the slope from 4 to 8 thousand? Then run the cell.

In [ ]:
print("slope from 1 to 2:", slope(accuracy, 1, 2), "points per thousand tickets")
print("slope from 2 to 4:", slope(accuracy, 2, 4), "points per thousand tickets")
print("slope from 4 to 8:", slope(accuracy, 4, 8), "points per thousand tickets")

> **Check.** You should see 12, 3 and 0.75 points per thousand tickets. The curve gets flatter. On a curve, the slope depends on **where** you measure it, and also on **how far** you look.

## 4. The slope at one point: the derivative

Take the point at 2 thousand tickets. A **secant** is a straight line through this point and a second point, a small gap to the right. The next cell makes the gap smaller and smaller, as in the lesson's table.

> **Predict.** The gap goes from 2 down to 0.01. Does the slope go up, go down or stay the same? Near which number does it stop? Then run the cell.

In [ ]:
point = 2
rows = []
for gap in [2, 1, 0.5, 0.1, 0.01]:
    second = point + gap
    rise = accuracy(second) - accuracy(point)
    rows.append([gap, second, round(accuracy(second), 3), round(rise, 3), round(rise / gap, 3)])
table(["Gap", "Second point", "Accuracy there", "Rise", "Slope = rise ÷ gap"], rows, digits=3)

> **Check.** You should see the slopes 3, 4, 4.8, 5.714 and 5.97. They come closer and closer to **6**. At that limit, the line only touches the curve at the point: it is the **tangent**.

The **derivative** is the slope of the tangent. Python cannot make the gap 0, because then it divides by 0. But a very small gap, such as 0.000001, gives almost the exact derivative. The next cell does this at 2, 4 and 8 thousand tickets, as in Priya's worked example. It draws each derivative as a text bar.

Run it. You should see 6, 1.5 and 0.375 points per thousand tickets.

In [ ]:
def derivative(f, x, gap=0.000001):
    """The slope of f at x, measured with a very small gap."""
    return slope(f, x, x + gap)

points = [2, 4, 8]
slopes = [round(derivative(accuracy, x), 3) for x in points]
bars([f"{x},000 tickets" for x in points], slopes)

At 2,000 tickets, the next thousand labelled tickets add about 6 points. At 8,000, they add less than half a point. That is why Omar decides to label up to about 4,000 tickets.

> **Warning.** A derivative describes a **small** change near the point. Run the cell: the derivative at 2 is 6, but the real gain from 2,000 to 3,000 tickets is only 4 points, because the curve bends.

In [ ]:
print("derivative at 2:          ", round(derivative(accuracy, 2), 3))
print("real gain from 2 to 3:    ", accuracy(3) - accuracy(2))

## 5. Guided practice: the slope at 4,000 tickets

> **Your turn.** Replace each `...` with a calculation. Use `accuracy(...)` and the `slope(...)` function from the cells above. Then run the check cell below it.

In [ ]:
point = 4

second_x = point + 2
second_y = accuracy(second_x)
rise = second_y - accuracy(point)
slope_gap_2 = slope(accuracy, point, point + 2)
slope_gap_001 = slope(accuracy, point, point + 0.01)

Run the check cell. Each line says "Check passed." or what to look at again.

In [ ]:
expect("the second point", second_x, 6)
expect("the accuracy at the second point", second_y, 90)
expect("the rise", rise, 2)
expect("the slope with a gap of 2", slope_gap_2, 1)
expect("the slope with a gap of 0.01", slope_gap_001, 1.496, tol=0.001)

With a gap of 0.01, the slope is very close to the tangent slope at 4 thousand tickets, 1.5. Write one sentence with its unit in your `c03-notes.md`, for example: "At 4,000 labelled tickets, accuracy rises by about 1.5 points for each extra thousand labelled tickets, for small changes."

## 6. Independent variation: the slope at 8,000 tickets

> **Your turn.** Calculate the slope between 8 and 10 thousand tickets, and the slope between 8 and 8.01 thousand tickets. Then run the check cell.

In [ ]:
slope_8_to_10 = slope(accuracy, 8, 10)
slope_8_to_801 = slope(accuracy, 8, 8.01)

In [ ]:
expect("the slope from 8 to 10", slope_8_to_10, 0.3, tol=0.001)
expect("the slope from 8 to 8.01", slope_8_to_801, 0.3745, tol=0.0005)

The tangent slope in the lesson's explorer is 0.375. Another 1,000 tickets add less than half an accuracy point, so 50 hours of agent time buy very little.

## 7. Failure case: a slope used far from where you measured it

A plan says: "At 2,000 labelled tickets, the slope is 6 points per thousand. So 5,000 more tickets will add 30 points."

> **Predict.** What accuracy does the plan promise at 7,000 tickets? What does the curve give? Then run the cell.

In [ ]:
plan = accuracy(2) + derivative(accuracy, 2) * 5
print("the plan promises:", round(plan, 1), "%")
print("the curve gives:  ", round(accuracy(7), 2), "%")

> **Check.** You should see 112.0% for the plan: an impossible accuracy. The curve gives 90.57%. The slope at 2,000 tickets says nothing reliable about 7,000 tickets.

**Fix:** use a derivative only for small changes. For a large change, take a small step, measure the slope again at the new point, and repeat. Training methods follow this rule, as you see in the next two lessons.

## Check your understanding and recap

Answer the **knowledge checks** at the end of the [lesson page](https://learning.nextia-ai.com/courses/math/m04/functions-and-change/). They count toward your certificate when you are signed in and enrolled. The notebook does not have them.

In this notebook you wrote the ticket model as a function and measured its slope: 6 minutes per hundred words, the same everywhere on the line. On the learning curve, the slope changes from point to point. A line through two points turns into the tangent when the gap shrinks, and the slope of the tangent is the derivative: 6, 1.5 and 0.375 points per thousand tickets at 2, 4 and 8 thousand.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Loss and gradients](https://learning.nextia-ai.com/courses/math/m04/loss-and-gradients/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/math/m04/functions-and-change/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield tickets are fictional and have no real people. The learning curve is illustrative.